In [ ]:
import sys
import os
import copy
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader

sys.path.append('..')

from src.data.dataset import CashedCustomDataset
from src.data.degredation import get_transforms
from src.models.mlp import Mlp

from custom.figure import mm, color
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Determine the project root and dataset directory
dataset = "cmnist"
dataset_dir = '../dataset'
percent = "0.5pct"

# Load the training and validation datasets using the pre-prepared transform
train_dataset = CashedCustomDataset(dataset, dataset_dir, split="train", percent=percent,
                                transform=get_transforms(dataset, blur=0, color=1, test=False, exclude_to_tensor=True))
train_degradation_dataset = CashedCustomDataset(dataset, dataset_dir, split="train", percent=percent,
                                            transform=get_transforms(dataset, blur=3, color=0, test=False, exclude_to_tensor=True))
test_align_dataset = CashedCustomDataset(dataset, dataset_dir, split="test-align", percent=percent,
                                    transform=get_transforms(dataset, blur=0, color=1, test=True, exclude_to_tensor=True))
test_conflict_dataset = CashedCustomDataset(dataset, dataset_dir, split="test-conflict", percent=percent,
                                    transform=get_transforms(dataset, blur=0, color=1, test=True, exclude_to_tensor=True))
test_dataset = CashedCustomDataset(dataset, dataset_dir, split="test", percent=percent,
                                transform=get_transforms(dataset, blur=0, color=1, test=True, exclude_to_tensor=True))

# print the number of samples in each dataset
print(f"Train Dataset Size: {len(train_dataset)}")
print(f"Train Degradation Dataset Size: {len(train_degradation_dataset)}")
print(f"Test Align Dataset Size: {len(test_align_dataset)}")
print(f"Test Conflict Dataset Size: {len(test_conflict_dataset)}")
print(f"Test Dataset Size: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
train_degradation_loader = DataLoader(train_degradation_dataset, batch_size=128, shuffle=True, num_workers=0)
test_align_loader = DataLoader(test_align_dataset, batch_size=128, shuffle=False, num_workers=0)
test_conflict_loader = DataLoader(test_conflict_dataset, batch_size=128, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

In [ ]:
dir = "250604i_cmnist_decoding"
num_net = 10

figure_dir = os.path.join("..", "figures", dir)
save_dir = os.path.join("..", "results", "250604a_cmnist_0.5p")
if not os.path.exists(figure_dir):
    os.makedirs(figure_dir)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

In [ ]:
structure = [28*28*3, 100, 100, 32, 10]

criterion = nn.CrossEntropyLoss()

In [ ]:
# reconstruction of original image from feature vector
import torchvision.transforms as T
from PIL import Image

class MLPDecoder(nn.Module):
    def __init__(self):
        super(MLPDecoder, self).__init__()
        self.fc1 = nn.Linear(100, 28*28*3)

    def forward(self, x):
        x = torch.sigmoid(self.fc1(x))
        return x

In [ ]:
class DecodingDataset(torch.utils.data.Dataset):
    def __init__(self, features, images, labels, attrs, transform=None):
        self.features = features
        self.images = images
        self.labels = labels
        self.attrs = attrs
        if transform is None:
            self.transform = T.Compose([T.ToTensor()])
        else:
            self.transform = transform

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        temp_image = self.images[idx]
        return self.features[idx], temp_image, self.labels[idx], self.attrs[idx]

In [ ]:
def measure_feature_vector(model, test_loader, target_layers):
    model_copy = copy.deepcopy(model).to(device)
    features_dict = {layer: [] for layer in target_layers}
    hooks = []
    for name, module in model_copy.named_modules():
        if name in target_layers:
            def hook_fn(module, input, output, key=name):
                features_dict[key].append(torch.flatten(output, 1).detach().cpu().numpy())
            hooks.append(module.register_forward_hook(hook_fn))

    labels_list, attrs_list = [], []

    model_copy.eval()
    with torch.no_grad():
        for inputs, labels, attrs in test_loader:
            inputs, labels, attrs = inputs.to(device), labels.to(device), attrs.to(device)
            _ = model_copy(inputs)
            labels_list.append(labels.cpu().numpy())
            attrs_list.append(attrs.cpu().numpy())

    for h in hooks:
        h.remove()

    for key in features_dict:
        features_dict[key] = np.concatenate(features_dict[key], axis=0)

    labels_array = np.concatenate(labels_list, axis=0)
    attrs_array = np.concatenate(attrs_list, axis=0)

    return features_dict, labels_array, attrs_array

In [ ]:
epoch = 40
num_net = 10
net_idx = 0

temp_model_wo = [Mlp(structure).to(device) for _ in range(num_net)]
temp_model_w = [Mlp(structure).to(device) for _ in range(num_net)]

for i in range(num_net):
    temp_model_wo[i].load_state_dict(torch.load(os.path.join(save_dir, f"model_wo_epoch_{i}_{epoch}.pth")))
    temp_model_w[i].load_state_dict(torch.load(os.path.join(save_dir, f"model_w_epoch_{i}_{epoch}.pth")))

In [ ]:
target_layer = "linear.1"

decoding_training_wo = [_ for _ in range(num_net)]
decoding_training_w = [_ for _ in range(num_net)]
decoding_test_w = [_ for _ in range(num_net)]
decoding_test_wo = [_ for _ in range(num_net)]

for net_idx in range(num_net):
    print(f"Processing network {net_idx + 1}/{num_net}")
    features_wo_align, labels_wo_align, attrs_wo_align = measure_feature_vector(temp_model_wo[net_idx], test_align_loader, [target_layer])
    features_w_align, labels_w_align, attrs_w_align = measure_feature_vector(temp_model_w[net_idx], test_align_loader, [target_layer])

    features_wo_align = np.array(features_wo_align[target_layer])
    features_w_align = np.array(features_w_align[target_layer])

    img_align = np.array([test_align_dataset.data[i][0].numpy() for i in range(len(test_align_dataset.data))])
    labels_align = np.array([test_align_dataset.data[i][1].numpy() for i in range(len(test_align_dataset.data))])
    attrs_align = np.array([test_align_dataset.data[i][2].numpy() for i in range(len(test_align_dataset.data))])

    features_wo_conflict, labels_wo_conflict, attrs_wo_conflict = measure_feature_vector(temp_model_wo[net_idx], test_conflict_loader, [target_layer])
    features_w_conflict, labels_w_conflict, attrs_w_conflict = measure_feature_vector(temp_model_w[net_idx], test_conflict_loader, [target_layer])

    features_wo_conflict = np.array(features_wo_conflict[target_layer])
    features_w_conflict = np.array(features_w_conflict[target_layer])

    img_conflict = np.array([test_conflict_dataset.data[i][0].numpy() for i in range(len(test_conflict_dataset.data))])
    labels_conflict = np.array([test_conflict_dataset.data[i][1].numpy() for i in range(len(test_conflict_dataset.data))])
    attrs_conflict = np.array([test_conflict_dataset.data[i][2].numpy() for i in range(len(test_conflict_dataset.data))])

    # conflict ratio in training set
    train_ratio = 0.9
    bias_ratio = 0.5/100

    num_align_train = int(len(img_align) * train_ratio)
    num_conflict_train = int(num_align_train * bias_ratio)

    print(f"Number of training samples from aligned data: {num_align_train}")
    print(f"Number of training samples from conflict data: {num_conflict_train}")

    print(f"Number of testing samples from aligned data: {len(img_align) - num_align_train}")
    print(f"Number of testing samples from conflict data: {len(img_conflict) - num_conflict_train}")

    # index
    idx_align_train = np.random.choice(len(img_align), num_align_train, replace=False)
    idx_align_test = np.setdiff1d(np.arange(len(img_align)), idx_align_train)
    idx_conflict_train = np.random.choice(len(img_conflict), num_conflict_train, replace=False)
    idx_conflict_test = np.setdiff1d(np.arange(len(img_conflict)), idx_conflict_train)

    features_wo_train = np.concatenate([features_wo_align[idx_align_train], features_wo_conflict[idx_conflict_train]])
    features_w_train = np.concatenate([features_w_align[idx_align_train], features_w_conflict[idx_conflict_train]])

    img_train = np.concatenate([img_align[idx_align_train], img_conflict[idx_conflict_train]])
    labels_train = np.concatenate([labels_align[idx_align_train], labels_conflict[idx_conflict_train]])
    attrs_train = np.concatenate([attrs_align[idx_align_train], attrs_conflict[idx_conflict_train]])

    features_wo_test = np.concatenate([features_wo_align[idx_align_test], features_wo_conflict[idx_conflict_test]])
    features_w_test = np.concatenate([features_w_align[idx_align_test], features_w_conflict[idx_conflict_test]])

    img_test = np.concatenate([img_align[idx_align_test], img_conflict[idx_conflict_test]])
    labels_test = np.concatenate([labels_align[idx_align_test], labels_conflict[idx_conflict_test]])
    attrs_test = np.concatenate([attrs_align[idx_align_test], attrs_conflict[idx_conflict_test]])


    decoding_training_wo[net_idx] = DecodingDataset(features_wo_train, img_train, labels_train, attrs_train)
    decoding_training_w[net_idx] = DecodingDataset(features_w_train, img_train, labels_train, attrs_train)
    decoding_test_wo[net_idx] = DecodingDataset(features_wo_test, img_test, labels_test, attrs_test)
    decoding_test_w[net_idx] = DecodingDataset(features_w_test, img_test, labels_test, attrs_test)

In [ ]:
# split decoding_test by bias_align and bias_conflict

def split_decoding_dataset(dataset):
    feature_align = []
    feature_conflict = []
    image_align = []
    image_conflict = []
    label_align = []
    label_conflict = []
    attr_align = []
    attr_conflict = []

    for i in range(len(dataset)):
        if dataset[i][2] == dataset[i][3]:
            feature_align.append(dataset[i][0])
            image_align.append(dataset[i][1])
            label_align.append(dataset[i][2])
            attr_align.append(dataset[i][3])
        else:
            feature_conflict.append(dataset[i][0])
            image_conflict.append(dataset[i][1])
            label_conflict.append(dataset[i][2])
            attr_conflict.append(dataset[i][3])

    dataset_align = DecodingDataset(np.array(feature_align), np.array(image_align), np.array(label_align), np.array(attr_align))
    dataset_conflict = DecodingDataset(np.array(feature_conflict), np.array(image_conflict), np.array(label_conflict), np.array(attr_conflict))
    return dataset_align, dataset_conflict

In [ ]:
decoding_training_loader_wo = [DataLoader(decoding_training_wo[net_idx], batch_size=128, shuffle=True, num_workers=0) for net_idx in range(num_net)]
decoding_training_loader_w = [DataLoader(decoding_training_w[net_idx], batch_size=128, shuffle=True, num_workers=0) for net_idx in range(num_net)]

decoding_test_loader_wo = [DataLoader(decoding_test_wo[net_idx], batch_size=128, shuffle=False, num_workers=0) for net_idx in range(num_net)]
decoding_test_loader_w = [DataLoader(decoding_test_w[net_idx], batch_size=128, shuffle=False, num_workers=0) for net_idx in range(num_net)]

In [ ]:
decoder_wo = [MLPDecoder().to(device) for _ in range(num_net)]
decoder_w = [MLPDecoder().to(device) for _ in range(num_net)]

optimizer_decoder_wo = [torch.optim.Adam(decoder_wo[net_idx].parameters(), lr=0.001, weight_decay=0.0001) for net_idx in range(num_net)]
optimizer_decoder_w = [torch.optim.Adam(decoder_w[net_idx].parameters(), lr=0.001, weight_decay=0.0001) for net_idx in range(num_net)]

criterion_decoder = nn.MSELoss()

training_info_decoder = {"train_loss" : [], "test_loss" : [], "best_net": None}
training_info_decoder_wo = [copy.deepcopy(training_info_decoder) for _ in range(num_net)]
training_info_decoder_w = [copy.deepcopy(training_info_decoder) for _ in range(num_net)]

In [ ]:
epochs = 100

# training decoder_wo
for net_idx in range(num_net):
    print(f"Training decoder w/o for network {net_idx + 1}/{num_net}")

    best_loss_wo = float('inf')
    best_net_decoder_wo = None

    for epoch in range(epochs):
        train_loss = 0.0
        decoder_wo[net_idx].train()
        for features, images, labels, attrs in decoding_training_loader_wo[net_idx]:
            features = features.to(device)
            images = images.to(device).view(-1, 3*28*28)
            optimizer_decoder_wo[net_idx].zero_grad()
            outputs = decoder_wo[net_idx](features)
            loss = criterion_decoder(outputs, images)
            loss.backward()
            optimizer_decoder_wo[net_idx].step()

            train_loss += loss.item()
        train_loss /= len(decoding_training_loader_wo[net_idx])
        test_loss = 0.0

        decoder_wo[net_idx].eval()
        with torch.no_grad():
            for features, images, labels, attrs in decoding_test_loader_wo[net_idx]:
                features = features.to(device)
                images = images.to(device).view(-1, 3*28*28)

                outputs = decoder_wo[net_idx](features)
                loss = criterion_decoder(outputs, images)

                test_loss += loss.item()
        test_loss /= len(decoding_test_loader_wo[net_idx])
        training_info_decoder_wo[net_idx]["train_loss"].append(train_loss)
        training_info_decoder_wo[net_idx]["test_loss"].append(test_loss)

        if test_loss < best_loss_wo:
            best_loss_wo = test_loss
            training_info_decoder_wo[net_idx]["best_net"] = copy.deepcopy(decoder_wo[net_idx].state_dict())

        print(f"Epoch {epoch+1}/{epochs} - Decoder w/o: Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

In [ ]:
# training decoder_w
for net_idx in range(num_net):
    print(f"Training decoder w for network {net_idx + 1}/{num_net}")

    best_loss_w = float('inf')
    best_net_decoder_w = None

    for epoch in range(epochs):
        train_loss = 0.0
        decoder_w[net_idx].train()
        for features, images, labels, attrs in decoding_training_loader_w[net_idx]:
            features = features.to(device)
            images = images.to(device).view(-1, 3*28*28)
            optimizer_decoder_w[net_idx].zero_grad()
            outputs = decoder_w[net_idx](features)
            loss = criterion_decoder(outputs, images)
            loss.backward()
            optimizer_decoder_w[net_idx].step()

            train_loss += loss.item()
        train_loss /= len(decoding_training_loader_w[net_idx])
        test_loss = 0.0

        decoder_w[net_idx].eval()
        with torch.no_grad():
            for features, images, labels, attrs in decoding_test_loader_w[net_idx]:
                features = features.to(device)
                images = images.to(device).view(-1, 3*28*28)

                outputs = decoder_w[net_idx](features)
                loss = criterion_decoder(outputs, images)

                test_loss += loss.item()
        test_loss /= len(decoding_test_loader_w[net_idx])
        training_info_decoder_w[net_idx]["train_loss"].append(train_loss)
        training_info_decoder_w[net_idx]["test_loss"].append(test_loss)

        if test_loss < best_loss_w:
            best_loss_w = test_loss
            training_info_decoder_w[net_idx]["best_net"] = copy.deepcopy(decoder_w[net_idx].state_dict())

        print(f"Epoch {epoch+1}/{epochs} - Decoder w: Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

In [ ]:
from custom.figure import plot_error

plt.figure(figsize=(40*mm, 30*mm))
data = [training_info_decoder_wo[net_idx]["train_loss"] for net_idx in range(num_net)]
plot_error(data, color=color["orange"])
data = [training_info_decoder_w[net_idx]["train_loss"] for net_idx in range(num_net)]
plot_error(data, color=color["sky"])
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.ylim([0.015, 0.03])
plt.title("Training Loss of Decoder")
plt.savefig(os.path.join(figure_dir, "decoder_training_loss.svg"))

In [ ]:
plt.figure(figsize=(40*mm, 30*mm))
data = [training_info_decoder_wo[net_idx]["test_loss"] for net_idx in range(num_net)]
plot_error(data, color=color["orange"])
data = [training_info_decoder_w[net_idx]["test_loss"] for net_idx in range(num_net)]
plot_error(data, color=color["sky"])
plt.ylim([0.024, 0.03])
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.savefig(os.path.join(figure_dir, "decoder_test_loss.svg"))

In [ ]:
# align and conflict test set
def test_decoding(decoder, test_loader, criterion, device):
    test_loss = 0.0
    decoder.eval()
    with torch.no_grad():
        for features, images, labels, attrs in test_loader:
            features = features.to(device)
            images = images.to(device).view(-1, 3*28*28)

            outputs = decoder(features)
            loss = criterion(outputs, images)

            test_loss += loss.item()
    test_loss /= len(test_loader)
    return test_loss

In [ ]:
# get the reconstructed image of the test set

def get_reconstructed_images(decoder, test_loader):
    decoder.eval()
    reconstructed_images_list = []
    original_images_list = []
    labels_list = []
    attrs_list = []
    with torch.no_grad():
        for features, images, labels, attrs in test_loader:
            features = features.to(device)
            images = images.to(device).view(-1, 3*28*28)
            outputs = decoder(features)
            reconstructed_images_list.append(outputs.cpu().numpy())
            original_images_list.append(images.cpu().numpy())
            labels_list.append(labels.cpu().numpy())
            attrs_list.append(attrs.cpu().numpy())
    reconstructed_images_list = np.concatenate(reconstructed_images_list, axis=0)
    original_images_list = np.concatenate(original_images_list, axis=0)
    labels_list = np.concatenate(labels_list, axis=0)
    attrs_list = np.concatenate(attrs_list, axis=0)
    return reconstructed_images_list, original_images_list, labels_list, attrs_list

reconstructed_images_wo, original_images_wo, labels_wo, attrs_wo = get_reconstructed_images(decoder_wo, decoding_test_loader_wo)
reconstructed_images_w, original_images_w, labels_w, attrs_w = get_reconstructed_images(decoder_w, decoding_test_loader_w)

In [ ]:
# find the index of the 10x10 grid (x: label, y: bias)
def get_grid_index(labels, attrs, grid_size=10):
    grid_indices = [[] for _ in range(grid_size)]
    for i in range(grid_size):
        for j in range(grid_size):
            index = np.where((labels == i) & (attrs == j))[0]
            if len(index) > 0:
                grid_indices[i].append(index)
    return grid_indices

grid_indices_wo = get_grid_index(labels_wo, attrs_wo, grid_size=10)
grid_indices_w = get_grid_index(labels_w, attrs_w, grid_size=10)

In [ ]:
# plot examples
plt.figure(figsize=(5, 5))
for i in range(10):
    for j in range(10):
        plt.subplot(10, 10, i*10+j+1)
        idx = grid_indices_w[i][j][0]
        original_image = original_images_w[idx].reshape(3, 28, 28).transpose(1, 2, 0)
        plt.imshow(original_image)
        plt.axis('off')
plt.tight_layout()

In [ ]:
# plot examples
for trial in range(10):
    plt.figure(figsize=(5, 5))
    for i in range(10):
        for j in range(10):
            plt.subplot(10, 10, i*10+j+1)
            idx = grid_indices_w[i][j][trial]
            recon_w = reconstructed_images_w[idx].reshape(3, 28, 28).transpose(1, 2, 0)
            plt.imshow(recon_w)
            plt.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, f"recon_w_{trial}.svg"))

In [ ]:
# plot examples
for trial in range(10):
    plt.figure(figsize=(5, 5))
    for i in range(10):
        for j in range(10):
            plt.subplot(10, 10, i*10+j+1)
            idx = grid_indices_wo[i][j][trial]
            recon_w = reconstructed_images_wo[idx].reshape(3, 28, 28).transpose(1, 2, 0)
            plt.imshow(recon_w)
            plt.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, f"recon_wo_{trial}.svg"))